# Test video trên Colab T4: Wan2.1-1.3B và LTX-Video 2B distilled

Mục tiêu: đo **thời gian**, **VRAM đỉnh** và lưu MP4 thật trên T4 16GB.

Lưu ý T4 (Turing sm75):
- Không có bf16, không có FlashAttention → dùng **fp16**, VAE **fp32**, offload qua CPU.
- T4 yếu hơn 4090 ~2.5–4 lần; clip 480p/3–5s là hợp lý.
- Nên chọn `Runtime → Change runtime type → T4 GPU`. Nếu load model OOM do RAM 12.7GB, dùng High-RAM hoặc giảm cấu hình theo ô troubleshooting cuối notebook.

Ước tính để so sánh: Wan1.3B 10–25 phút/clip; LTX 2B distilled 2–6 phút/clip (sẽ có số thật sau khi chạy).

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("HF_HOME", "/content/hf")

import subprocess, sys, time, gc, json
import torch

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
print("python", sys.version.split()[0], "| torch", torch.__version__, "| cuda", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Chưa bật GPU: Runtime → Change runtime type → T4 GPU")

In [ ]:
# Không cài lại torch (Colab đã có bản khớp CUDA).
!pip -q install --upgrade "diffusers>=0.35" "transformers>=4.45" accelerate safetensors ftfy imageio imageio-ffmpeg bitsandbytes

In [ ]:
from pathlib import Path
from diffusers.utils import export_to_video

VID = Path("/content/videos")
VID.mkdir(exist_ok=True)
RESULTS = []

def run_bench(name, fn, fps):
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    try:
        frames = fn()
        sec = time.time() - t0
        peak = torch.cuda.max_memory_allocated() / 2**30
        out = VID / f"{name}.mp4"
        export_to_video(frames, str(out), fps=fps)
        RESULTS.append({"model": name, "sec": round(sec, 1), "peak_vram_gb": round(peak, 2),
                        "frames": len(frames), "fps": fps, "file": str(out)})
        print(f"[OK] {name}: {sec:.1f}s | peak VRAM {peak:.2f} GB | {len(frames)} frames → {out}")
        return frames
    except torch.cuda.OutOfMemoryError as exc:
        torch.cuda.empty_cache()
        print(f"[OOM] {name}: {exc}")
        RESULTS.append({"model": name, "error": "OOM", "detail": str(exc)[:200]})
        return None

PROMPT = ("A cinematic shot of a red tram moving through a rainy Hanoi street at dusk, "
          "warm street lights reflecting on wet asphalt, people with umbrellas, slow camera pan.")
NEG = "worst quality, blurry, jittery, distorted, static, watermark, text"
print("helpers ready")

## 1) Wan2.1 T2V-1.3B

- Repo diffusers chính thức: `Wan-AI/Wan2.1-T2V-1.3B-Diffusers`.
- Text encoder UMT5-XXL rất lớn; notebook thử **8-bit** trước để vừa RAM Colab, fallback fp16.
- Cấu hình test: 832×480, 49 frame (~3s @16fps), 30 steps, guidance 6 (khuyến nghị cho 1.3B).

In [ ]:
from diffusers import WanPipeline, AutoencoderKLWan

WAN_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
wan_vae = AutoencoderKLWan.from_pretrained(WAN_ID, subfolder="vae", dtype=torch.float32)

try:
    from diffusers.quantizers import PipelineQuantizationConfig
    qc = PipelineQuantizationConfig(
        quant_backend="bitsandbytes_8bit",
        quant_kwargs={"load_in_8bit": True},
        components_to_quantize=["text_encoder"],
    )
    wan = WanPipeline.from_pretrained(WAN_ID, vae=wan_vae, quantization_config=qc, dtype=torch.float16)
    print("Wan: text encoder 8-bit")
except Exception as exc:
    print("8-bit text encoder không dùng được, fallback fp16:", type(exc).__name__, str(exc)[:200])
    gc.collect(); torch.cuda.empty_cache()
    wan = WanPipeline.from_pretrained(WAN_ID, vae=wan_vae, dtype=torch.float16)

wan.enable_model_cpu_offload()
try:
    wan.vae.enable_tiling()
except Exception:
    pass
print("Wan pipeline ready")

In [ ]:
def gen_wan():
    return wan(
        prompt=PROMPT,
        negative_prompt=NEG,
        height=480,
        width=832,
        num_frames=49,
        num_inference_steps=30,
        guidance_scale=6.0,
    ).frames[0]

run_bench("wan21_1.3b_480p_49f", gen_wan, fps=16)

In [ ]:
# Nếu ô trên OOM: chạy ô này (33 frame, 384p) để lấy số thật thay vì bỏ cuộc.
def gen_wan_small():
    return wan(
        prompt=PROMPT,
        negative_prompt=NEG,
        height=384,
        width=640,
        num_frames=33,
        num_inference_steps=25,
        guidance_scale=6.0,
    ).frames[0]

run_bench("wan21_1.3b_384p_33f", gen_wan_small, fps=16)

In [ ]:
# Giải phóng VRAM/RAM trước khi load model thứ hai.
del wan
gc.collect()
torch.cuda.empty_cache()
print("freed Wan")

## 2) LTX-Video 2B 0.9.8 distilled

- Transformer lấy từ single-file `ltxv-2b-0.9.8-distilled.safetensors` (6.34GB).
- T5 + VAE + tokenizer lấy từ repo `Lightricks/LTX-Video` (không tải transformer 13B).
- Distilled: `guidance_scale=1.0`, 8 steps; 704×480, 121 frame (~5s @24fps).

In [ ]:
from diffusers import LTXPipeline, AutoModel, AutoencoderKLLTXVideo, FlowMatchEulerDiscreteScheduler
from transformers import T5EncoderModel, T5TokenizerFast

LTX_BASE = "Lightricks/LTX-Video"
LTX_SINGLE = "https://huggingface.co/Lightricks/LTX-Video/resolve/main/ltxv-2b-0.9.8-distilled.safetensors"

ltx_tokenizer = T5TokenizerFast.from_pretrained(LTX_BASE, subfolder="tokenizer")
ltx_text_encoder = T5EncoderModel.from_pretrained(LTX_BASE, subfolder="text_encoder", dtype=torch.float16)
ltx_vae = AutoencoderKLLTXVideo.from_pretrained(LTX_BASE, subfolder="vae", dtype=torch.float32)
ltx_transformer = AutoModel.from_single_file(LTX_SINGLE, dtype=torch.float16)
ltx_scheduler = FlowMatchEulerDiscreteScheduler.from_pretrained(LTX_BASE, subfolder="scheduler")

ltx = LTXPipeline(
    tokenizer=ltx_tokenizer,
    text_encoder=ltx_text_encoder,
    vae=ltx_vae,
    scheduler=ltx_scheduler,
    transformer=ltx_transformer,
)
ltx.enable_model_cpu_offload()
try:
    ltx.vae.enable_tiling()
except Exception:
    pass
print("LTX pipeline ready")

In [ ]:
def gen_ltx():
    return ltx(
        prompt=PROMPT,
        negative_prompt=NEG,
        width=704,
        height=480,
        num_frames=121,
        num_inference_steps=8,
        guidance_scale=1.0,
        decode_timestep=0.05,
        decode_noise_scale=0.025,
    ).frames[0]

run_bench("ltx_2b_distilled_480p_121f", gen_ltx, fps=24)

In [ ]:
# Nếu OOM: 97 frame, 512×384.
def gen_ltx_small():
    return ltx(
        prompt=PROMPT,
        negative_prompt=NEG,
        width=512,
        height=384,
        num_frames=97,
        num_inference_steps=8,
        guidance_scale=1.0,
        decode_timestep=0.05,
        decode_noise_scale=0.025,
    ).frames[0]

run_bench("ltx_2b_distilled_384p_97f", gen_ltx_small, fps=24)

In [ ]:
# Tổng hợp kết quả + lưu JSON.
import pandas as pd
df = pd.DataFrame(RESULTS)
display(df)
Path("/content/videos/results.json").write_text(json.dumps(RESULTS, indent=2))
print("Files:", [str(p) for p in VID.glob("*.mp4")])

# Nếu muốn giữ video qua session (Colab free sẽ xoá máy):
# from google.colab import drive; drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/video-tests && cp /content/videos/*.mp4 /content/drive/MyDrive/video-tests/

## Troubleshooting

- **OOM khi load**: dùng Runtime High-RAM, hoặc bỏ 8-bit fallback, hoặc chạy ComfyUI với weights fp8/GGUF.
- **OOM khi generate**: giảm `num_frames` trước (49→33, 121→97), sau đó giảm height/width; giữ chiều rộng chia hết cho 32.
- **Video đen/Nan**: fp16 trên T4 có thể lỗi ở VAE; giữ VAE fp32 (đã set), hoặc decode bằng `output_type="latent"` rồi giải mã riêng.
- **Hết dung lượng**: `rm -rf ~/.cache/pip /content/hf/hub` hoặc đặt `HF_HOME=/content/hf` (đã set).
- **Colab ngắt session**: job >2h dễ chết; mỗi session chỉ nên test 2–4 clip.
- **Số liệu T4**: ghi lại `sec`, `peak_vram_gb` trong bảng cuối để so với ước tính.

Nguồn tham khảo:
- Wan2.1: https://github.com/Wan-Video/Wan2.1
- Wan 1.3B: https://huggingface.co/Wan-AI/Wan2.1-T2V-1.3B
- LTX-Video: https://github.com/Lightricks/LTX-Video
- Diffusers LTX: https://huggingface.co/docs/diffusers/main/en/api/pipelines/ltx_video